In [6]:
import pandas as pd
import json
import os
import numpy as np
from PIL import Image

In [7]:
df = pd.read_csv(r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\data\df_overall.csv").fillna("")
df.head()

,Unnamed: 0,U_id,Location Category,Location,CT_image_paths,MRI_image_paths,title,history,findings,case diagnosis,Differential Diagnosis,captions,combined_text
0,0,MPX1009,Reproductive and Urinary System,Genitourinary,['C:\\fyp_manish_shyam_phase2\\MedPix-2.0-main...,[],Bladder Diverticulum,73-year-old male with hematuria and numerous w...,Bladder with thickened wall and diverticulum o...,Bladder Diverticulum,Bladder Diverticulum,The prostate is enlarged with several calcific...,Bladder with thickened wall and diverticulum o...
1,1,MPX1024,Thorax,"Chest, Pulmonary",['C:\\fyp_manish_shyam_phase2\\MedPix-2.0-main...,[],Left upper lobe collapse caused by an enlargin...,60-year-old woman presents with chest pain and...,• PA chest radiograph demonstrates left lung v...,Left upper lobe collapse caused by an enlargin...,This combination of radiographic findings are ...,CT of the chest reveals an obstructing mass an...,• PA chest radiograph demonstrates left lung v...
2,2,MPX1012,Reproductive and Urinary System,Genitourinary,['C:\\fyp_manish_shyam_phase2\\MedPix-2.0-main...,[],Ovarian torsion,"24 hours of pelvic, RLQ pain.",CT: Large heterogeneous pelvic mass displacing...,Ovarian torsion,"Ovarian Torsion Ovarian mass, benign vs. malig...",pelvis,CT: Large heterogeneous pelvic mass displacing...
3,3,MPX1016,Thorax,"Chest, Pulmonary",['C:\\fyp_manish_shyam_phase2\\MedPix-2.0-main...,[],Adenocarcinoma of the Lung,The patient is a 43-year-old female who presen...,Chest PA/LAT revealed increased interstitial m...,Adenocarcinoma of the Lung,lymphangitic spread of malignancy primary mali...,Contrast enhanced chest CT shows diffuse incre...,Chest PA/LAT revealed increased interstitial m...
4,4,MPX1035,Head,Head and Neck,['C:\\fyp_manish_shyam_phase2\\MedPix-2.0-main...,[],Occipital Condyle Fractures,22yo M transported to the emergency department...,Axial and coronal CT of the head and cervical ...,Occipital Condyle Fractures,"Occipital condylar fracture Type I, II, or III",Non-contrast CT axial view demonstrates a R oc...,Axial and coronal CT of the head and cervical ...


In [15]:
for _ in range(20): print("\n\n",df["combined_text"][_])



 Bladder with thickened wall and diverticulum on the right. Diverticulum is mostly likely secondary to chronic outflow obstruction. Prostate enlargement. The prostate is enlarged with several calcifications  noted within.  No dominant prostate mass is evident. Bladder is prominent with mildly thickened wall. There is a small posteriolateral diverticulum on the rightward aspect.


 • PA chest radiograph demonstrates left lung volume loss, silhouetting of the left cardiac border, and Luftsichel sign. • Lateral chest radiograph shows anterior displacement of the major fissure and elevation of the left mainstem bronchus. • CT of the chest reveals an obstructing mass and resultant LUL collapse. CT of the chest reveals an obstructing mass and resultant LUL collapse.


 CT: Large heterogeneous pelvic mass displacing the uterus anteriorly. US: Enlarged right ovary with central cystic component. Absent blood flow. pelvis


 Chest PA/LAT revealed increased interstitial markings in the right lo

In [16]:
import re

def clean_medical_report(text: str) -> str:
    if not isinstance(text, str):
        return ""

    t = text

    # -----------------------------
    # 1. Normalize unicode bullets & dashes
    # -----------------------------
    t = t.replace("•", " ")
    t = t.replace("–", "-").replace("—", "-")

    # -----------------------------
    # 2. Remove Figure / Image references
    # -----------------------------
    # (Fig 1), (Fig. 2:3), Figure 4, Image 2:
    t = re.sub(r"\(?(fig|figure|image)\.?\s*\d+[:.]?\d*\)?", " ", t, flags=re.I)
    t = re.sub(r"(fig|figure|image)\.?\s*\d+[:.]?\d*", " ", t, flags=re.I)

    # -----------------------------
    # 3. Remove bullet numbering (1., 2), 3:)
    # -----------------------------
    t = re.sub(r"(^|\n|\s)\d+[\.\):\-]\s*", " ", t)

    # -----------------------------
    # 4. Remove repeated modality headers
    #    (keep first occurrence)
    # -----------------------------
    modality_patterns = [
        r"\bCT\b\s*:",
        r"\bMRI\b\s*:",
        r"\bUS\b\s*:",
        r"\bPA/LAT\b\s*:",
        r"\bPA/Lat\b\s*:",
        r"\bCXR\b\s*:",
    ]

    for pat in modality_patterns:
        t = re.sub(pat, "", t, count=1, flags=re.I)  # keep first, remove rest

    # -----------------------------
    # 5. Remove generic image descriptors
    # -----------------------------
    t = re.sub(
        r"(axial|coronal|sagittal)\s+(ct|mri)\s+(image|images|view|views)",
        " ",
        t,
        flags=re.I
    )

    # -----------------------------
    # 6. Collapse excessive punctuation
    # -----------------------------
    t = re.sub(r"[,:;]{2,}", ", ", t)
    t = re.sub(r"\.{2,}", ".", t)
    t = re.sub(r"\-\-+", "-", t)

    # -----------------------------
    # 7. Remove duplicated sentences (very important)
    # -----------------------------
    sentences = [s.strip() for s in re.split(r"\.\s*", t) if len(s.strip()) > 3]
    seen = set()
    unique = []

    for s in sentences:
        key = s.lower()
        if key not in seen:
            unique.append(s)
            seen.add(key)

    t = ". ".join(unique)

    # -----------------------------
    # 8. Whitespace cleanup
    # -----------------------------
    t = re.sub(r"\s+", " ", t).strip()

    return t


In [17]:
df["combined_clean"] = df["combined_text"].apply(clean_medical_report)


In [19]:
for _ in range(len(df)): print("\n\n",df["combined_clean"][_])



 Bladder with thickened wall and diverticulum on the right. Diverticulum is mostly likely secondary to chronic outflow obstruction. Prostate enlargement. The prostate is enlarged with several calcifications noted within. No dominant prostate mass is evident. Bladder is prominent with mildly thickened wall. There is a small posteriolateral diverticulum on the rightward aspect


 PA chest radiograph demonstrates left lung volume loss, silhouetting of the left cardiac border, and Luftsichel sign. Lateral chest radiograph shows anterior displacement of the major fissure and elevation of the left mainstem bronchus. CT of the chest reveals an obstructing mass and resultant LUL collapse


 Large heterogeneous pelvic mass displacing the uterus anteriorly. Enlarged right ovary with central cystic component. Absent blood flow. pelvis


 Chest PA/LAT revealed increased interstitial markings in the right lower lobe. Contrast enhanced chest CT revealed diffuse increased interstitial markings invo

In [20]:
MODALITY_REGEX = re.compile(
    r"""
    (?<!\w)                          # not preceded by a word char
    (
        (?:CT|MRI|US|XRAY|X[-\s]?RAY|
         RADIOGRAPH|RADIOGRAPHS|
         CXR|PA/LAT|PA\\LAT|PA-LAT|
         PA|LAT|
         PET|PET-CT|
         SPECT|
         ULTRASOUND|
         MAMMOGRAM|
         ANGIOGRAPHY|
         FLUOROSCOPY|
         HRCT)
        (?:\s*\([^)]+\))?            # optional parentheses: (contrast)
    )
    \s*[:\-–]                        # delimiter
    """,
    re.IGNORECASE | re.VERBOSE
)


In [21]:
def extract_modalities(text):
    if not isinstance(text, str):
        return []
    return [m.group(1).strip() for m in MODALITY_REGEX.finditer(text)]


In [25]:
all_modalities = []

for txt in df["combined_clean"]:
    all_modalities.extend(extract_modalities(txt))


In [26]:
from collections import Counter

modality_counts = Counter(all_modalities)

for modality, count in modality_counts.most_common():
    print(f"{modality:25s} -> {count}")


CT                        -> 25
MRI                       -> 9
CXR                       -> 5
Ultrasound                -> 3
PET                       -> 3
radiograph                -> 2
PA                        -> 2
US                        -> 2
Radiograph                -> 2
x-ray                     -> 2
Radiographs               -> 2
MRI (T1W)                 -> 2
CT (with contrast)        -> 1
CT (from emergency dept)  -> 1
MRI (from emergency dept) -> 1
CT (following worsening left sided weakness and sensory loss) -> 1
MRI (following worsening left sided weakness and sensory loss) -> 1
Lat                       -> 1
PA/LAT                    -> 1
X-Ray                     -> 1
CXR (AP Portable)         -> 1
radiographs               -> 1
ultrasound                -> 1
radiograph (skull)        -> 1
X-ray                     -> 1
RADIOGRAPHS               -> 1
CT (Sag recon)            -> 1
MRI (Sag )                -> 1
MRI (Sagital STIR)        -> 1
RADIOGRAPH                -

In [31]:
colons = []
for rep in df["combined_clean"]:
    if ":" in rep: colons.append(rep)
        
print(len(colons))
for rep in colons: print("\n\n", rep)

126


 smoothly marginated soft tissue opacity noted in Right cardiophrenic angle, otherwise normal. homogenous fluid attenuating and smoothly marginated lesion abutting the right cardiac border with Hounsfield attenuation unit = 7; Measures 5 x 4 x 7 cm CT: homogenous fluid attenuating and smoothly marginated lesion abutting the right cardiac border with Hounsfield attenuation unit = 7; Measures 5 x 4 x 7 cm


 CT scan abdomen: Positive “arrowhead sign” in appendix. Presence of thickened appendix. Questionable appendicolith. Terminal ileum normal and seen with contrast, Good visualization of ileocecal juntion. Cecum normal size without evidence of edema. CT scan abdomen: Positive “arrowhead sign” in appendix. Terminal ileum normal and seen with contrast, Good visualization of ileocecal juntion


 PA/Lat There is irregularity to the contour of the right hilum on the PA view. On the lateral view, there is a central, RML nodule which when going back to the PA, can barely be seen. Axial N

In [28]:
show_examples("CT", n=3)
show_examples("RADIOGRAPH", n=3)
show_examples("PA/LAT", n=3)



=== Examples for: CT ===

[Example 1]
Bladder with thickened wall and diverticulum on the right. Diverticulum is mostly likely secondary to chronic outflow obstruction. Prostate enlargement.

[Example 2]
• PA chest radiograph demonstrates left lung volume loss, silhouetting of the left cardiac border, and Luftsichel sign. • Lateral chest radiograph shows anterior displacement of the major fissure and elevation of the left mainstem bronchus. • CT of the chest reveals an obstructing mass and resultant LUL collapse.

[Example 3]
CT: Large heterogeneous pelvic mass displacing the uterus anteriorly. US: Enlarged right ovary with central cystic component. Absent blood flow.


=== Examples for: RADIOGRAPH ===

[Example 1]
• PA chest radiograph demonstrates left lung volume loss, silhouetting of the left cardiac border, and Luftsichel sign. • Lateral chest radiograph shows anterior displacement of the major fissure and elevation of the left mainstem bronchus. • CT of the chest reveals an obst

In [32]:
#cleaning colons, urls, headings, etc.

import re
import unicodedata


In [33]:
# URLs & citations
URL_REGEX = re.compile(r"http\S+|www\S+|PMID:\s*\d+", re.IGNORECASE)

# Bullet points & symbols
BULLET_REGEX = re.compile(r"[•►▪■●◦→▶]+|\*{1,3}|>>+")

# Dates / admin junk
DATE_REGEX = re.compile(
    r"\b(?:\d{1,2}\s+[A-Za-z]{3,9}\s+\d{4}|Hospital Day\s*\d+|Admission)\b",
    re.IGNORECASE
)

# Modality / section headers (ONLY when followed by colon)
MODALITY_HEADER_REGEX = re.compile(
    r"""
    (?<!\w)
    (
        CT|MRI|MR|US|ULTRASOUND|XRAY|X[-\s]?RAY|
        RADIOGRAPH|RADIOGRAPHY|CXR|PA/LAT|PA-LAT|
        PET|PET-CT|HRCT|
        CHEST|ABDOMEN|PELVIS|HEAD|BRAIN|
        CT\s+SCAN|CT\s+CHEST|CT\s+ABDOMEN|
        MRI\s+BRAIN|MRI\s+SPINE
    )
    (?:\s*\([^)]+\))?
    \s*:
    """,
    re.IGNORECASE | re.VERBOSE
)

# Image / figure references
IMAGE_REF_REGEX = re.compile(
    r"""
    (?<!\w)
    (
        Image\s*\d+|
        Film\s*#?\d+|
        Axial|Coronal|Sagittal|
        T1|T2|FLAIR|DWI|ADC|
        Fat\s*Sat|STIR|
        Recon|Reformation
    )
    \s*(?:view|image|images)?\s*:
    """,
    re.IGNORECASE | re.VERBOSE
)

# Excess punctuation
PUNCT_CLEAN_REGEX = re.compile(r"[;]{2,}|[,]{3,}|[.]{4,}")


In [34]:
def deduplicate_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    seen = set()
    cleaned = []
    for s in sentences:
        key = s.lower().strip()
        if key and key not in seen:
            seen.add(key)
            cleaned.append(s)
    return " ".join(cleaned)


In [35]:
def clean_medical_report(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Normalize unicode
    text = unicodedata.normalize("NFKC", text)

    # Remove URLs / PMIDs
    text = URL_REGEX.sub("", text)

    # Remove bullets
    text = BULLET_REGEX.sub(" ", text)

    # Remove dates / admin
    text = DATE_REGEX.sub("", text)

    # Remove modality headers
    text = MODALITY_HEADER_REGEX.sub("", text)

    # Remove image / figure references
    text = IMAGE_REF_REGEX.sub("", text)

    # Clean excessive punctuation
    text = PUNCT_CLEAN_REGEX.sub(".", text)

    # Deduplicate sentences
    text = deduplicate_sentences(text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [37]:
df["combined_clean"] = df["findings"].apply(clean_medical_report)

# Quick sanity check
for i in range(3):
    print("RAW:\n", df["findings"].iloc[i][:500])
    print("\nCLEAN:\n", df["combined_clean"].iloc[i][:500])
    print("="*80)


RAW:
 Bladder with thickened wall and diverticulum on the right. Diverticulum is mostly likely secondary to chronic outflow obstruction. Prostate enlargement.

CLEAN:
 Bladder with thickened wall and diverticulum on the right. Diverticulum is mostly likely secondary to chronic outflow obstruction. Prostate enlargement.
RAW:
 • PA chest radiograph demonstrates left lung volume loss, silhouetting of the left cardiac border, and Luftsichel sign. • Lateral chest radiograph shows anterior displacement of the major fissure and elevation of the left mainstem bronchus. • CT of the chest reveals an obstructing mass and resultant LUL collapse.

CLEAN:
 PA chest radiograph demonstrates left lung volume loss, silhouetting of the left cardiac border, and Luftsichel sign. Lateral chest radiograph shows anterior displacement of the major fissure and elevation of the left mainstem bronchus. CT of the chest reveals an obstructing mass and resultant LUL collapse.
RAW:
 CT: Large heterogeneous pelvic mas

In [40]:
colons = []
for rep in df["combined_clean"]:
    if ":" in rep: colons.append(rep)
        
print(len(colons))
for rep in colons: print("\n\n", rep)

93


 ABDOMINAL AORTA: There is narrowing of the abdominal aorta, both above and below the renal arteries. The superior mesenteric artery is occluded at its origin, and not seen on the lateral view. There is a large Arc of Riolan from the IMA, which reconstitutes the SMA distribution. The celiac axis is patent, however, there is a 50% stenosis at the origin. There are single renal arteries bilaterally, both of which demonstrate stenoses. On the right, there is a long segment stenosis with approximately 50% narrowing. On the left, there is 60% to 70% stenosis at the origin of the vessel, extending to an early bifurcation, with an early upper pole branch. This is also stenotic at its origin.


 RUQ a heterogenous mass in the region of the head of the pancreas with hepatic biliary ductal dilation. CT Chest/Abd/Pelvis w/contrast: multiple ground glass opacities bilaterally in lungs, large mass about the pancreatic head which surrounds the periaortic tissues and celiac axis. PET scan: focal

In [41]:
SEMANTIC_HEADERS = [
    "findings",
    "impression",
    "conclusion",
    "summary",
    "diagnosis",
    "history",
    "indication",
    "comparison",
    "technique",
    "abdominal aorta",
    "neoplasm name",
    "us findings",
    "ct findings",
    "mri findings",
    "radiologic findings",
    "imaging findings"
]


In [42]:
JUNK_COLON_HEADER_REGEX = re.compile(
    r"""
    (?<!\w)
    (
        figure\s*\d+|
        film\s*#?\d+|
        image\s*\d+|
        images?\s*\d+(?:,\d+)*|
        cxr\d+|
        \d+(?:st|nd|rd|th)\s+image|
        \d+\s+jan|
        scout
    )
    \s*:
    """,
    re.IGNORECASE | re.VERBOSE
)


In [43]:
def normalize_semantic_colons(text):
    for header in SEMANTIC_HEADERS:
        pattern = re.compile(
            rf"\b{re.escape(header)}\s*:",
            re.IGNORECASE
        )
        text = pattern.sub(f"{header.capitalize()}.", text)
    return text


In [44]:
def remove_residual_colons(text):
    # Replace colon with period only if surrounded by whitespace
    text = re.sub(r"\s:\s", ". ", text)
    return text


In [45]:
def clean_medical_report_v2(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)

    text = URL_REGEX.sub("", text)
    text = BULLET_REGEX.sub(" ", text)
    text = DATE_REGEX.sub("", text)

    text = MODALITY_HEADER_REGEX.sub("", text)
    text = IMAGE_REF_REGEX.sub("", text)

    text = JUNK_COLON_HEADER_REGEX.sub("", text)
    text = normalize_semantic_colons(text)
    text = remove_residual_colons(text)

    text = PUNCT_CLEAN_REGEX.sub(".", text)
    text = deduplicate_sentences(text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


In [46]:
df["combined_clean"] = df["combined_clean"].apply(clean_medical_report_v2)

In [47]:
remaining = df["combined_clean"].str.contains(":", na=False).sum()
print("Remaining colons:", remaining)


Remaining colons: 69


In [48]:
colons = []
for rep in df["combined_clean"]:
    if ":" in rep: colons.append(rep)
        
print(len(colons))
for rep in colons: print("\n\n", rep)

69


 RUQ a heterogenous mass in the region of the head of the pancreas with hepatic biliary ductal dilation. CT Chest/Abd/Pelvis w/contrast: multiple ground glass opacities bilaterally in lungs, large mass about the pancreatic head which surrounds the periaortic tissues and celiac axis. PET scan: focal increased uptake about the pancreatic head, scattered foci bilateral lungs L>R


 Abdominal 1. Mild stranding focally in the lesser sac, medial to the second portion of the duodenum, anterior to the third portion of the duodenum, posterior to the pancreas and just inferior to the pancreatic head, with mild stranding adjacent to the SMA.2. Possible mild narrowing of the third portion of the duodenum between the SMA and aorta.Addendum: Small thrombus within the proximal portion of the SMA. Distal to this region the contrast is not as bright as the proximal SMA, with mild adjacent inflammatory changes adjacent to the SMA consistent with SMA Syndrome.


 Chest and abdominal CT on revealed t

In [55]:
BLIND_COLON_HEADERS = [
    "pet scan",
    "addendum",
    "ct liver",
    "ct w/out contrast",
    "radiograph of l-spine",
    "month male",
    "year old female",
    "rads",
    "head ct in our hospital 5 hours later",
    "t1wi",
    "next 3 images",
    "last image",
    "film #1",
    "film #2 and 3",
    "film #4",
    "ct scans",
    "radiographs",
    "patient 1",
    "patient 2",
    "plain film",
    "initial study",
    "chest pa",
    "cect demonstrates",
    "plain films",
    "radiology",
    "diagnostic angiogram",
    "t2 ax",
    "t1 ax +c",
    "t1 cor +c",
    "t1 cor",
    "t1 sag",
    "a-p radiographs",
    "mri w/ contrast",
    "note",
    "sonography",
    "cxr dol 0"
]


In [56]:
BLIND_HEADER_REGEX = re.compile(
    r"""
    (?<!\w)
    (
        """ + "|".join(re.escape(h) for h in BLIND_COLON_HEADERS) + r"""
    )
    \s*:
    """,
    re.IGNORECASE | re.VERBOSE
)


In [57]:
def remove_blind_headers(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return BLIND_HEADER_REGEX.sub("", text)


In [58]:
df["combined_clean"] = df["combined_clean"].apply(remove_blind_headers)

In [59]:
remaining = df["combined_clean"].str.contains(":", na=False).sum()
print("Remaining colons:", remaining)


Remaining colons: 47


In [60]:
colons = []
for rep in df["combined_clean"]:
    if ":" in rep: colons.append(rep)
        
print(len(colons))
for rep in colons: print("\n\n", rep)

47


 RUQ a heterogenous mass in the region of the head of the pancreas with hepatic biliary ductal dilation. CT Chest/Abd/Pelvis w/contrast: multiple ground glass opacities bilaterally in lungs, large mass about the pancreatic head which surrounds the periaortic tissues and celiac axis.  focal increased uptake about the pancreatic head, scattered foci bilateral lungs L>R


 Chest and abdominal CT on revealed the following in comparison with an August 2004 chest and abdominal Progression of right hilar adenopathy and enlargement of the pleural-based soft tissue mass along the posteromedial aspect of the right lower lobe. Pulmonary parenchyma demonstrates enlargement of multiple right lower lobe pulmonary nodules. The pancreas, spleen, adrenal glands, and kidneys are normal. Soft tissue in the bilateral gluteal regions are normal. Chest and abdomal CT on after 4 months of chemotherapy and radiation revealed the following in comparison with the study: Subcarinal lymphadenopathy is demons

In [61]:
# now blindly remove all the :
def remove_all_colons(text: str) -> str:
    if not isinstance(text, str):
        return ""
    
    # Replace colon with period to preserve sentence boundary
    text = text.replace(":", "")
    
    # Fix duplicated punctuation
    text = re.sub(r"\.\s*\.", ".", text)
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


In [62]:
df["combined_clean"] = df["combined_clean"].apply(remove_all_colons)


In [63]:
remaining = df["combined_clean"].str.contains(":", na=False).sum()
print("Remaining colons:", remaining)


Remaining colons: 0


In [65]:
semis = []
for rep in df["combined_clean"]:
    if ";" in rep: semis.append(rep)
        
print(len(semis))
for rep in semis: print("\n\n", rep)

18


 smoothly marginated soft tissue opacity noted in Right cardiophrenic angle, otherwise normal. homogenous fluid attenuating and smoothly marginated lesion abutting the right cardiac border with Hounsfield attenuation unit = 7; Measures 5 x 4.4 x 7 cm


 Head Bilateral Lacrimal gland enlargement with homogenous tissue attenuation. Multiple lucenies of the skull bilaterally, particularly posterior to the vertex ranging 5-9 mm in size. Diffuse soft tissue attenuation noted in right maxillary sinus consistent with sinusitis. Skeletal Survey Left upper extremity-single round, punched-out appearing lucencies in the region of the radial tuberosity and proximal humerus; PA-chest-single, round, punched out appearing lucency in the lateral left clavicle; Skull-numerous, round, punched-out appearing lucencies over the parietal and frontal bones.


 Chest radiography demonstrates dextrocardia with the cardiac apex pointing to the right. There is a right-sided aortic arch, associated with slig

In [66]:
# replacing all ";" with ""

def remove_all_semis(text: str) -> str:
    if not isinstance(text, str):
        return ""
    
    # Replace colon with period to preserve sentence boundary
    text = text.replace(";", "")
    
    # Fix duplicated punctuation
    text = re.sub(r"\.\s*\.", ".", text)
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


In [67]:
df["combined_clean"] = df["combined_clean"].apply(remove_all_semis)


In [68]:
remaining = df["combined_clean"].str.contains(";", na=False).sum()
print("Remaining semis:", remaining)


Remaining semis: 0


In [69]:
for rep in df["combined_clean"]:
    print("\n\n", rep)



 Bladder with thickened wall and diverticulum on the right. Diverticulum is mostly likely secondary to chronic outflow obstruction. Prostate enlargement.


 PA chest radiograph demonstrates left lung volume loss, silhouetting of the left cardiac border, and Luftsichel sign. Lateral chest radiograph shows anterior displacement of the major fissure and elevation of the left mainstem bronchus. CT of the chest reveals an obstructing mass and resultant LUL collapse.


 Large heterogeneous pelvic mass displacing the uterus anteriorly. Enlarged right ovary with central cystic component. Absent blood flow.


 Chest PA/LAT revealed increased interstitial markings in the right lower lobe. Contrast enhanced chest CT revealed diffuse increased interstitial markings involving the right middle and lower lobes, pleural thickening/scarring of the posterior right lower lobe, and a small right-sided pleural effusion.


 Axial and coronal CT of the head and cervical spine demonstrating a R Occipital co

In [70]:
dashes = []
for rep in df["combined_clean"]:
    if "-" in rep: dashes.append(rep)
        
print(len(dashes))
for rep in dashes: print("\n\n", rep)

297


 Chest PA/LAT revealed increased interstitial markings in the right lower lobe. Contrast enhanced chest CT revealed diffuse increased interstitial markings involving the right middle and lower lobes, pleural thickening/scarring of the posterior right lower lobe, and a small right-sided pleural effusion.


 - Cholesteatoma of the left mesotympanum and epitympanum - Extends medially toward the malleus and incus without clear erosion of the bones - No dehiscence of the facial nerve noted


 The frontal and lateral views of the thoracic spine demonstrate a mixed lucent/sclerotic appearance of the T12 vertebral body and the left pedicle. A sagittal view MRI with gadolinium of the thoracic spine and lumbar spine demonstrates enhancing lesions multiple vertebrae, including T12. Soft tissue enhancement is also present from T9-L2 paraspinal muscles. A sagittal nonconstrast CT obtained several weeks later demonstrates worsening expansile lytic lesions to T10, T11, and T12. MRI of the thora

In [74]:
def remove_dangling_dashes(text: str) -> str:
    text = re.sub(r"\s-\s", " ", text)
    text = re.sub(r"(?m)^\s*-\s*", "", text)
    text = re.sub(r"\s*-\s*$", "", text)
    return text


In [75]:
df["combined_clean"] = df["combined_clean"].apply(remove_dangling_dashes)


In [76]:
dashes = []
for rep in df["combined_clean"]:
    if "-" in rep: dashes.append(rep)
        
print(len(dashes))
for rep in dashes: print("\n\n", rep)

280


 Chest PA/LAT revealed increased interstitial markings in the right lower lobe. Contrast enhanced chest CT revealed diffuse increased interstitial markings involving the right middle and lower lobes, pleural thickening/scarring of the posterior right lower lobe, and a small right-sided pleural effusion.


 The frontal and lateral views of the thoracic spine demonstrate a mixed lucent/sclerotic appearance of the T12 vertebral body and the left pedicle. A sagittal view MRI with gadolinium of the thoracic spine and lumbar spine demonstrates enhancing lesions multiple vertebrae, including T12. Soft tissue enhancement is also present from T9-L2 paraspinal muscles. A sagittal nonconstrast CT obtained several weeks later demonstrates worsening expansile lytic lesions to T10, T11, and T12. MRI of the thoracic spine was obtained five months after the patient began treatment. Interval worsening present at multiple levels, including multiple compression deformities and enhancing mass with c

In [78]:
for rep in df["combined_clean"]:
    print("\n\n" + rep)



Bladder with thickened wall and diverticulum on the right. Diverticulum is mostly likely secondary to chronic outflow obstruction. Prostate enlargement.


PA chest radiograph demonstrates left lung volume loss, silhouetting of the left cardiac border, and Luftsichel sign. Lateral chest radiograph shows anterior displacement of the major fissure and elevation of the left mainstem bronchus. CT of the chest reveals an obstructing mass and resultant LUL collapse.


Large heterogeneous pelvic mass displacing the uterus anteriorly. Enlarged right ovary with central cystic component. Absent blood flow.


Chest PA/LAT revealed increased interstitial markings in the right lower lobe. Contrast enhanced chest CT revealed diffuse increased interstitial markings involving the right middle and lower lobes, pleural thickening/scarring of the posterior right lower lobe, and a small right-sided pleural effusion.


Axial and coronal CT of the head and cervical spine demonstrating a R Occipital condyle

In [79]:
import re

def remove_numeric_bullets(text: str) -> str:
    """
    Removes numeric bullet points like '1.', '2.', etc.
    Preserves decimal numbers like 2.6, 3.1, etc.
    """
    # remove numeric bullets (not decimals)
    text = re.sub(r"\b\d+\.(?!\d)", "", text)

    # normalize extra spaces
    text = re.sub(r"\s{2,}", " ", text)

    return text.strip()


In [80]:
df["combined_clean"] = df["combined_clean"].apply(remove_numeric_bullets)


In [81]:
for rep in df["combined_clean"]:
    if "1." in rep: print("\n\n" + rep)



Parenchymal bridge ("isthmus") connecting the inferior poles of both kidneys = Horseshoe kidneys Complete duplication of the right ureter. a 0.7 x 0.3 cm fat containing lesion in the inferior pole of the left kidney. Multiple tiny calcifications bilaterally. 1.3cm cyst in the anterior aspect of the left kidney, containing a small calcification on its edge.


Non-contrast CT images of the brain demonstrate a 1.3 x 1.2 cm mass within the left optic nerve with multiple additional diffuse hemorrhagic lesions with associated vasogenic edema and mass effect. The largest and most concerning mass is within the right frontal lobe. This mass measures approximately 2.9 cm in greatest dimension. A whole body bone scan demonstrates intense focal uptake at the left femoral head, and subtle intake within the mid right femoral diaphysis.


Skull cap within the left lower anterior abdominal wall. There is a 10x6x1.5 cm nonenhancing fluid collection just posterior to the skull cap.


Skull Series Ther

In [82]:
for rep in df["combined_clean"]:
    print("\n\n" + rep)



Bladder with thickened wall and diverticulum on the right. Diverticulum is mostly likely secondary to chronic outflow obstruction. Prostate enlargement.


PA chest radiograph demonstrates left lung volume loss, silhouetting of the left cardiac border, and Luftsichel sign. Lateral chest radiograph shows anterior displacement of the major fissure and elevation of the left mainstem bronchus. CT of the chest reveals an obstructing mass and resultant LUL collapse.


Large heterogeneous pelvic mass displacing the uterus anteriorly. Enlarged right ovary with central cystic component. Absent blood flow.


Chest PA/LAT revealed increased interstitial markings in the right lower lobe. Contrast enhanced chest CT revealed diffuse increased interstitial markings involving the right middle and lower lobes, pleural thickening/scarring of the posterior right lower lobe, and a small right-sided pleural effusion.


Axial and coronal CT of the head and cervical spine demonstrating a R Occipital condyle

In [84]:
# now, checking the differential diagnosis column
for dd in df["Differential Diagnosis"]:
    print("\n\n" + dd)



Bladder Diverticulum


This combination of radiographic findings are consistent with LUL collapse and highly suspicious for an underlying endobronchial mass causing obstruction of the LUL bronchus.


Ovarian Torsion Ovarian mass, benign vs. malignant Hemorrhagic cyst Ectopic pregnancy TOA


lymphangitic spread of malignancy primary malignancy lymphoma Sjogrens syndrome lymphangioleiomyomatosis pneumonia (bacterial, atypical, viral) collagen vascular disease (SLE, RA) hypersensitivity pneumonitis asbestosis edema (secondary to fluid overload, CHF, or nephrotic syndrome)


Occipital condylar fracture Type I, II, or III


DDX is that of ILD (see FACTOID). *Pulmonary edema (CHF) *Bacterial pneumonia *Pulmonary alveolar proteinosis


Congenital Cholesteatoma Acquired Cholesteatoma Giant Cholesterol Cyst Acoustic Neuroma Glomus tumor Sarcoma Meningioma


• Pericardial cyst • Morgagni hernia • Cardiac (epicardial) fat pad • Adenopathy • Thymoma • Lymphoma


Fracture


• Horseshoe kidney • R

In [85]:
df.to_csv(r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\data\df_overall.csv")

In [87]:
dashes = []
for rep in df["combined_clean"]:
    if "- "  in rep or " -" in rep or " - " in rep: dashes.append(rep)
        
print(len(dashes))
for rep in dashes: print("\n\n", rep)

31


 CT/ There is vertebra plana of L5, with edema and enhancement of the remaining marrow into the posterior elements. There is displacement of the thecal sac posteriorly by the extruded bony fragments. -RADIOGRAPH/ Left upper lobe lung mass with numerous bilateral pulmonary nodules, consistent with metastatic disease.


 CXR dextrocardia, bronchus intermedius on the left side of the chest KUB- Liver edge on the left, gastric bubble on the right Abd CT- complete mirror image of all abdominal organs. No evidence of nephrolithiasis on non-contrast CT


 Contrast head CT is significant for a multiloculated cystic posterior fossa midline mass that is adjacent to the torcula with a portion extending into the occipital calvarium. This mass is low in density with HU measuring There is rim enhancement with several internal septations. There is distortion of the 4th. ventricle with associated dilatation of the temporal horns. Bone windows demonstrate smooth calvarial remodeling of the occiput

In [88]:
# blindly remove all dashes that satisfy the above condition

df["combined_clean"] = (
    df["combined_clean"]
    .str.replace(" - ", " ", regex=False)
    .str.replace("- ", " ", regex=False)
    .str.replace(" -", " ", regex=False)
)


In [90]:
dashes = []
for rep in df["combined_clean"]:
    if "-"in rep: dashes.append(rep)
        
print(len(dashes))
for rep in dashes: print("\n\n", rep)

266


 Chest PA/LAT revealed increased interstitial markings in the right lower lobe. Contrast enhanced chest CT revealed diffuse increased interstitial markings involving the right middle and lower lobes, pleural thickening/scarring of the posterior right lower lobe, and a small right-sided pleural effusion.


 The frontal and lateral views of the thoracic spine demonstrate a mixed lucent/sclerotic appearance of the T12 vertebral body and the left pedicle. A sagittal view MRI with gadolinium of the thoracic spine and lumbar spine demonstrates enhancing lesions multiple vertebrae, including T12. Soft tissue enhancement is also present from T9-L2 paraspinal muscles. A sagittal nonconstrast CT obtained several weeks later demonstrates worsening expansile lytic lesions to T10, T11, and T12. MRI of the thoracic spine was obtained five months after the patient began treatment. Interval worsening present at multiple levels, including multiple compression deformities and enhancing mass with c

In [91]:
import re

dashes = []
pattern = re.compile(r"[A-Za-z]+-[A-Za-z]+")

for rep in df["combined_clean"]:
    if pattern.search(rep):
        dashes.append(rep)

print(len(dashes))
for rep in dashes:
    print("\n\n", rep)


243


 Chest PA/LAT revealed increased interstitial markings in the right lower lobe. Contrast enhanced chest CT revealed diffuse increased interstitial markings involving the right middle and lower lobes, pleural thickening/scarring of the posterior right lower lobe, and a small right-sided pleural effusion.


 Axial CT with intravenous and oral contrast material demonstrates a large fluid collection predominantly within the porta hepatis around multiple surgical clips consistent with prior cholecystectomy. More fluid is seen around the right lobe of the liver and more inferiorly within the right paracolic gutter. Peritoneal fat around the fluid collection demonstrates stranding suggestive of inflammation. Planar and SPECT images after the administration of Tc99m-disofenin IV demonstrates large amount of abnormal radiotracer accumulation within the area corresponding to the large fluid collection seen on the CT images, consistent with a biliary leak. Note that a tubular area of radiot

In [92]:
df["combined_clean"] = df["combined_clean"].str.replace(
    r"(?<=[A-Za-z])-(?=[A-Za-z])",
    " ",
    regex=True
)


In [93]:
dashes = []
for rep in df["combined_clean"]:
    if "-"in rep: dashes.append(rep)
        
print(len(dashes))
for rep in dashes: print("\n\n", rep)

70


 The frontal and lateral views of the thoracic spine demonstrate a mixed lucent/sclerotic appearance of the T12 vertebral body and the left pedicle. A sagittal view MRI with gadolinium of the thoracic spine and lumbar spine demonstrates enhancing lesions multiple vertebrae, including T12. Soft tissue enhancement is also present from T9-L2 paraspinal muscles. A sagittal nonconstrast CT obtained several weeks later demonstrates worsening expansile lytic lesions to T10, T11, and T12. MRI of the thoracic spine was obtained five months after the patient began treatment. Interval worsening present at multiple levels, including multiple compression deformities and enhancing mass with cortical disruption and contiguous anterior soft tissue mass lifting the anterior longitudinal ligament. Involvment of the posterior elements is present with narrowing of multiple neural foramina and spinal cord compression.


 4-5 mm calcified stone in the expected region of the right submandibular gland du

In [94]:
dashes = []
for rep in df["combined_clean"]:
    if "--"in rep: dashes.append(rep)
        
print(len(dashes))
for rep in dashes: print("\n\n", rep)

3


 Compression fracture involving the posterior aspect of L1 with posterior displacement of the fracture fragment--concern for burst fracture L1 burst fracture Known polycystic kidney disease incidentally imaged


 NON CONTRAST Head CT (from emergency dept) regional hypoattenuation in right temporal lobe, extending from cerebral cortex into white matter loss of gray/white matter differentiation sulcal effacement of right temporal parietal lobes (may represent ischemia and resultant edema) some chronic microvascular ischemic disease of periventricular white matter ventricles and cisterns preserved, no intracranial hemorrhage or mass likely represents infarct of unknown time course (subacute vs. acute) MRI recommended Brain MRI (from emergency dept) axial DWI—increased signal in right posterior parietal lobe=restricted diffusion ADC map--corresponding decreased signal consistent with acute stroke in right posterior parietal lobe CAROTID US echogenic calcified plaque in right carotid bu

In [95]:
df["combined_clean"] = df["combined_clean"].str.replace(
    r"-{2,}",
    " ",
    regex=True
)

In [97]:
dashes = []
for rep in df["combined_clean"]:
    if "-"in rep: dashes.append(rep)
        
print(len(dashes))
for rep in dashes: print("\n\n", rep)

68


 The frontal and lateral views of the thoracic spine demonstrate a mixed lucent/sclerotic appearance of the T12 vertebral body and the left pedicle. A sagittal view MRI with gadolinium of the thoracic spine and lumbar spine demonstrates enhancing lesions multiple vertebrae, including T12. Soft tissue enhancement is also present from T9-L2 paraspinal muscles. A sagittal nonconstrast CT obtained several weeks later demonstrates worsening expansile lytic lesions to T10, T11, and T12. MRI of the thoracic spine was obtained five months after the patient began treatment. Interval worsening present at multiple levels, including multiple compression deformities and enhancing mass with cortical disruption and contiguous anterior soft tissue mass lifting the anterior longitudinal ligament. Involvment of the posterior elements is present with narrowing of multiple neural foramina and spinal cord compression.


 4-5 mm calcified stone in the expected region of the right submandibular gland du

In [102]:
dashes = []
for rep in df["combined_clean"]:
    if "fig" in rep.lower(): dashes.append(rep)
        
print(len(dashes))
for rep in dashes: print("\n\n", rep)

16


 Axial CT without contrast The noncontrast head CT revealed a small hyperdense subdural hematoma, consistent with the acute injury (Fig. 1a). There is pneumocephalus, seen as small bubbles of air within the left extraaxial parietal subdural collection as well as a single focus of air posteriorly at the left paramidline aspect of the extraaxial space shown (Fig. These findings indicate that a fracture has occurred, allowing communication of outside air or a sinus cavity with the intracranial space. Indeed a complex adjacent temporal bone fracture was detected and is best appreciated using bone windows (Fig. 1b).


 CT scan (without contrast due to renal insufficiency) revealed no evidence of intra abdominal abscess, however the liver was found to harbor multiple lesions, biopsy of which revealed metastatic transitional cell carcinoma (figure 1). Additonally, a lytic lesion was found of the L2 vertebral body (figure 2). Because of the recurring fevers an Indium-111 tagged WBC study 

In [107]:
dashes = []
for rep in df["combined_clean"]:
    if "fig" in rep.lower(): dashes.append(rep)
        
print(len(dashes))
for rep in dashes: print("\n\n", rep)

16


 Axial CT without contrast The noncontrast head CT revealed a small hyperdense subdural hematoma, consistent with the acute injury (Fig. 1a). There is pneumocephalus, seen as small bubbles of air within the left extraaxial parietal subdural collection as well as a single focus of air posteriorly at the left paramidline aspect of the extraaxial space shown (Fig. These findings indicate that a fracture has occurred, allowing communication of outside air or a sinus cavity with the intracranial space. Indeed a complex adjacent temporal bone fracture was detected and is best appreciated using bone windows (Fig. 1b).


 CT scan (without contrast due to renal insufficiency) revealed no evidence of intra abdominal abscess, however the liver was found to harbor multiple lesions, biopsy of which revealed metastatic transitional cell carcinoma (figure 1). Additonally, a lytic lesion was found of the L2 vertebral body (figure 2). Because of the recurring fevers an Indium-111 tagged WBC study 

In [110]:
df["combined_clean"] = df["combined_clean"].str.replace(
    r"""(?ix)
    \(?                         # optional opening parenthesis
    \bfig(?:ure)?s?             # fig, figure, figs, figures
    \.?                         # optional dot
    \s*                         # optional whitespace
    (?:                         # optional figure identifiers
        [A-Za-z]                # A, B, C
        |
        \d+[A-Za-z]?            # 1, 1a, 2b
        |
        \d+\s*(?:and|&)\s*\d+   # 2 and 3, 3&4
        |
        [A-Za-z]\s*(?:and|&)\s*[A-Za-z]  # A and B
    )*
    \)?                         # optional closing parenthesis
    """,
    "",
    regex=True
)


In [118]:
quotes = []
for rep in df["combined_clean"]:
    if "\"" in rep.lower(): quotes.append(rep)
        
print(len(quotes))
for rep in quotes: print("\n\n", rep)

30


 Parenchymal bridge ("isthmus") connecting the inferior poles of both kidneys = Horseshoe kidneys Complete duplication of the right ureter. a 0.7 x 0.3 cm fat containing lesion in the inferior pole of the left kidney. Multiple tiny calcifications bilaterally. 1.3cm cyst in the anterior aspect of the left kidney, containing a small calcification on its edge.


 Contrast enhanced axial CT images of the abdomen demonstrates the presence of a "whirlpool sign" inversion of the superior mesenteric vessels all of the small bowel loops on the right, left sided colon, and inflamatory changes within the appendix consistent with phlegmon.


 "Bipolar" mass involving the petrous bone and posterior fossa Destructive petrous (temporal bone) mass posteromedial, near vestibular aqueduct extends to middle ear erosion includes right sigmoid area heterogeneous mixed cystic and solid hyperintense on both T1W and T2W proteinaceous and/or hemorrhagic fluid


 The chest x ray was "negative" and is not i

In [119]:
bullets = []
for rep in df["combined_clean"]:
    if "(" in rep.lower(): bullets.append(rep)
        
print(len(bullets))
for rep in bullets: print("\n\n", rep)

116


 Parenchymal bridge ("isthmus") connecting the inferior poles of both kidneys = Horseshoe kidneys Complete duplication of the right ureter. a 0.7 x 0.3 cm fat containing lesion in the inferior pole of the left kidney. Multiple tiny calcifications bilaterally. 1.3cm cyst in the anterior aspect of the left kidney, containing a small calcification on its edge.


 Selected Images – CT (contrast, arterial phase) of abdomen/pelvis Large, diffusely infiltrated fatty liver with accessory left lobe. Compare to density of spleen. Multiple areas of focal sparing in left lobe that appears nodular. Area of focal sparing near portal vein. Area focal sparing in left lobe. Renal cyst and renal calculi.


 RUQ A large mass is seen in the right upper quadrant in the expected placement of the right kidney. The mass appears heterogeneous with multiple cystic components. The mass appears somewhat well marginated. No hypervascularity is seen. A wedge of renal parenchyma can be seen at the interface of

In [120]:
# removing paranthesises
df["combined_clean"] = df["combined_clean"].str.replace(r"[()]", "", regex=True)


In [121]:
df["combined_clean"] = (
    df["combined_clean"]
    .str.replace(r"\s{2,}", " ", regex=True)
    .str.strip()
)
# cleanup to remove extra spaces

In [126]:
bullets = []
for rep in df["combined_clean"]:
    if "\"" in rep: bullets.append(rep)
        
print(len(bullets))
for rep in bullets: print("\n\n", rep)

30


 Parenchymal bridge "isthmus" connecting the inferior poles of both kidneys = Horseshoe kidneys Complete duplication of the right ureter. a 0.7 x 0.3 cm fat containing lesion in the inferior pole of the left kidney. Multiple tiny calcifications bilaterally. 1.3cm cyst in the anterior aspect of the left kidney, containing a small calcification on its edge.


 Contrast enhanced axial CT images of the abdomen demonstrates the presence of a "whirlpool sign" inversion of the superior mesenteric vessels all of the small bowel loops on the right, left sided colon, and inflamatory changes within the appendix consistent with phlegmon.


 "Bipolar" mass involving the petrous bone and posterior fossa Destructive petrous temporal bone mass posteromedial, near vestibular aqueduct extends to middle ear erosion includes right sigmoid area heterogeneous mixed cystic and solid hyperintense on both T1W and T2W proteinaceous and/or hemorrhagic fluid


 The chest x ray was "negative" and is not inclu

In [127]:
df["combined_clean"] = df["combined_clean"].str.replace('"', '', regex=False)


In [130]:
bullets = []
for rep in df["combined_clean"]:
    if "“" in rep or "”" in rep: bullets.append(rep)
        
print(len(bullets))
for rep in bullets: print("\n\n", rep)

14


 CT scan Positive “arrowhead sign” in appendix. Presence of thickened appendix. Questionable appendicolith. Terminal ileum normal and seen with contrast, Good visualization of ileocecal juntion. Cecum normal size without evidence of edema.


 proximal partial bowel obstruction dilated loops of small bowel with CT oral contrast in colon NGT Thickened segment of distal small bowel with luminal narrowing Focal fatty proliferation “creeping fat”


 The MRI/MRA study showed no vascular pathology, but incidentally it was noted that there was a heterogeneous and mildly expanded clivus. Whole body skeletal scintigraphy showed nonspecific increased uptake in the clivus region. Repeat CT of the sinus was performed, revealing a “ground glass appearance”, containing a smaller circumscribed area, which has the appearance of a cystic lesion. Overall the process appeared nonaggressive and was stable on 5 month follow up CT.


 Outside head CT reported small 1cm homogenously round hyperdense lesi

In [131]:
df["combined_clean"] = df["combined_clean"].str.replace('“', '', regex=False)
df["combined_clean"] = df["combined_clean"].str.replace('”', '', regex=False)


In [137]:
bullets = []
for rep in df["combined_clean"]:
    if "," in rep: bullets.append(rep)
        
print(len(bullets))
for rep in bullets: print("\n\n", rep)

382


 PA chest radiograph demonstrates left lung volume loss, silhouetting of the left cardiac border, and Luftsichel sign. Lateral chest radiograph shows anterior displacement of the major fissure and elevation of the left mainstem bronchus. CT of the chest reveals an obstructing mass and resultant LUL collapse.


 Chest PA/LAT revealed increased interstitial markings in the right lower lobe. Contrast enhanced chest CT revealed diffuse increased interstitial markings involving the right middle and lower lobes, pleural thickening/scarring of the posterior right lower lobe, and a small right sided pleural effusion.


 Predominately basilar and peripheral interlobular septal thickening with scattered areas of ground glass opacity, consolidation, and fibrosis.


 smoothly marginated soft tissue opacity noted in Right cardiophrenic angle, otherwise normal. homogenous fluid attenuating and smoothly marginated lesion abutting the right cardiac border with Hounsfield attenuation unit = 7 Mea

In [140]:
bullets = []
for rep in df["combined_clean"]:
    if "#" in rep: bullets.append(rep)
        
print(len(bullets))
for rep in bullets: print("\n\n", rep)

6


 RUQ A large mass is seen in the right upper quadrant in the expected placement of the right kidney. The mass appears heterogeneous with multiple cystic components. The mass appears somewhat well marginated. No hypervascularity is seen. A wedge of renal parenchyma can be seen at the interface of the tumor and the liver on slide # Abd. A 14cm, heterogenous renal mass extending from the right kidney with surrounding rim of renal parenchyma. There are multiple low density collections within the mass. There is no evidence of fat within the mass or calcifications suggesting against renal angiomyolipoma or rhabdoid tumor of the kidney respectively. Both kidneys demonstrate contrast enhancement and excretion. No obvious tumor extensions into the renal vein or inferior vena cava is noted. Mildly enlarged mesenteric lymph nodes adjacent to the kidney suggest possible malignant spread or may be reactive.


 The parenchyma of the lung demonstrates bilateral upper lobe predominant fibrotic cha

In [141]:
df["combined_clean"] = df["combined_clean"].str.replace("#", "", regex=False)


In [142]:
df["combined_clean"] = (
    df["combined_clean"]
    .str.replace(r"\s{2,}", " ", regex=True)
    .str.strip()
)


In [147]:
bullets = []
for rep in df["combined_clean"]:
    if "–" in rep: bullets.append(rep)
        
print(len(bullets))
for rep in bullets: print("\n\n", rep)

5


 Selected Images – CT contrast, arterial phase of abdomen/pelvis Large, diffusely infiltrated fatty liver with accessory left lobe. Compare to density of spleen. Multiple areas of focal sparing in left lobe that appears nodular. Area of focal sparing near portal vein. Area focal sparing in left lobe. Renal cyst and renal calculi.


 AP radiograph of the abdomen reveals multiple 2 – 3mm punctate ovoid sclerotic foci in the femurs and pelvis, clustered in a periarticular distribution. CT of pelvis demonstrates multiple punctate, oblong sclerotic foci symmetrically distributed throughout the proximal feumurs and pelvis, clustered in a periarticular distribution. No aggressive features are noted.


 CXR – increased opacity over lower T spine on lateral film CT – spiculated mass ~2cm in RLL c reticular stranding in contact c pleura no chest wall invasion PET – increased radiotracer uptake along posterior aspect of R mediastinum in region of R atrium


 MRI– Diffusion weighted images – I

In [148]:
df["combined_clean"] = df["combined_clean"].str.replace("–", "", regex=False)
df["combined_clean"] = (
    df["combined_clean"]
    .str.replace(r"\s{2,}", " ", regex=True)
    .str.strip()
)


In [152]:
df.to_csv(r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\data\df_overall.csv")